# Fetching macroeconomic data from the ECB Data Portal

**Purpose.** This notebook shows, end to end, how to fetch macroeconomic time
series — history *and* official forecasts — from the **European Central Bank
Data Portal** ([data.ecb.europa.eu](https://data.ecb.europa.eu)). We focus on
the ECB **only**: every number below comes from one public API, with no
authentication and no other data provider.

We cover **15 European countries** (Austria, Belgium, Denmark, France,
Germany, Ireland, Italy, Luxembourg, Netherlands, Norway, Poland, Portugal,
Spain, Sweden, United Kingdom) and 17 macro-financial variables: activity,
prices, labour market, housing, interest rates, credit, exchange rates,
fiscal position and sovereign stress.

**Structure**

- **Section A** — the important datasets, with direct links to their official
  series catalogues (CSV);
- **Section B** — extracting the *labels* of all relevant series for our
  countries (no data yet), ending in a pinned, hardcoded label list;
- **Section C** — downloading every series into pandas;
- **Section D** — statistics on what we downloaded, and the official
  projections from the current year onward.

**Requirements:** `pip install requests pandas matplotlib` — nothing else.
The notebook is fully self-contained: no local files are read or written.


In [1]:
import csv
import io
import re
import time
import zipfile
from datetime import date

import matplotlib.pyplot as plt
import pandas as pd
import requests

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 110)

API = "https://data-api.ecb.europa.eu/service/data"      # data endpoint
PORTAL = "https://data.ecb.europa.eu"                     # human-facing portal
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "ecb-macro-notebook/1.0"
CURRENT_YEAR = date.today().year

def show(df, caption=""):
    """Left-aligned display for public readability (no centered tables)."""
    sty = (df.style.set_caption(caption)
           .set_properties(**{"text-align": "left"})
           .set_table_styles([
               {"selector": "th", "props": "text-align: left; font-weight: bold;"},
               {"selector": "caption",
                "props": "text-align: left; font-size: 1.05em; font-weight: bold; padding-bottom: 6px;"}]))
    return sty

COUNTRIES = ['AT',
 'BE',
 'DE',
 'DK',
 'ES',
 'FR',
 'GB',
 'IE',
 'IT',
 'LU',
 'NL',
 'NO',
 'PL',
 'PT',
 'SE']
print(f"{len(COUNTRIES)} countries: {', '.join(COUNTRIES)}")
print(f"current year: {CURRENT_YEAR}")

15 countries: AT, BE, DE, DK, ES, FR, GB, IE, IT, LU, NL, NO, PL, PT, SE
current year: 2026


---
## A — The datasets

The ECB Data Portal hosts ~100 datasets ("dataflows"). The twelve used here
cover country-level macro-credit work. Two carry *forecasts*: **MPD** holds
the Eurosystem/ECB staff projections, and **AMECO** series end with ~2 years
of European Commission forecasts.

Every dataset page has a **Data information → Catalog** section offering the
dataset's *series catalogue*: a zipped CSV listing **every series** with its
key, first/last date, official title and dimension values. Those catalogues
are the ECB's own data dictionaries — section B validates against them.

- **`MNA`** — National accounts — real and nominal GDP *(history)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/MNA) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/MNA/download)
- **`ICP`** — HICP — consumer price inflation *(history)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/ICP) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/ICP/download)
- **`LFSI`** — Labour force survey — unemployment rate *(history)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/LFSI) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/LFSI/download)
- **`RPP`** — Residential property prices *(history)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/RPP) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/RPP/download)
- **`IRS`** — 10-year government bond yields *(history)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/IRS) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/IRS/download)
- **`FM`** — Financial markets — 3M EURIBOR *(history)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/FM) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/FM/download)
- **`MIR`** — Bank lending rates *(history, euro area)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/MIR) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/MIR/download)
- **`BSI`** — Bank balance sheets — credit stocks *(history)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/BSI) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/BSI/download)
- **`EXR`** — Euro foreign-exchange reference rates *(history)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/EXR) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/EXR/download)
- **`CISS`** — Systemic stress indicators *(history)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/CISS) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/CISS/download)
- **`MPD`** — Macroeconomic Projection Database *(forecasts)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/MPD) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/MPD/download)
- **`AME`** — AMECO annual macro & fiscal, incl. EC forecasts *(history + forecasts)*  
  [dataset page](https://data.ecb.europa.eu/data/datasets/AME) · [series catalogue (CSV, zipped)](https://data.ecb.europa.eu/data/datasets/AME/download)

General links:
[dataset catalogue](https://data.ecb.europa.eu/data/datasets) ·
[API documentation](https://data.ecb.europa.eu/help/api/overview) ·
machine-readable dataflow list: `https://data-api.ecb.europa.eu/service/dataflow/ECB`

*Footnote:* `MNA` and `LFSI` are Eurostat-sourced datasets served through the
ECB portal (SDMX agency `ESTAT`) — same API, same catalogue mechanism.


### What the variable labels mean

Short labels like `lending_rate_hh` are used throughout for readability. The
guide below is the dictionary: every label with its meaning, units, source
dataset and country coverage. It is also available as the `VARIABLE_GUIDE`
DataFrame for programmatic use.


In [2]:
VARIABLE_GUIDE = pd.DataFrame([{'countries': 'all 15',
  'dataset': 'MNA',
  'meaning': 'Real gross domestic product, chain-linked volume (ESA 2010, reference year '
             '2020), seasonally and calendar adjusted. National accounts main aggregate; '
             'enters models as growth (T4/T6).',
  'units': 'EUR millions, chain-linked volume',
  'variable': 'gdp_real'},
 {'countries': 'all 15',
  'dataset': 'ICP',
  'meaning': 'HICP inflation — Harmonised Index of Consumer Prices, all items, annual rate of '
             'change. The EU-harmonised consumer inflation measure.',
  'units': 'percent per annum (annual rate of change)',
  'variable': 'hicp'},
 {'countries': 'all 15',
  'dataset': 'LFSI',
  'meaning': 'Unemployment rate, ILO definition, ages 15-74, total, seasonally adjusted '
             '(Eurostat Labour Force Survey via ECB LFSI).',
  'units': 'percent of labour force',
  'variable': 'unemployment'},
 {'countries': 'all 15',
  'dataset': 'RPP',
  'meaning': 'Residential property prices, all types of dwellings (new and existing), whole '
             'country, index. ECB RPP indicator compiled from national/BIS sources (source '
             'code 4).',
  'units': 'index',
  'variable': 'house_prices'},
 {'countries': 'all 15',
  'dataset': 'IRS',
  'meaning': '10-year government bond yield — long-term interest rate for EU convergence '
             'purposes (Maastricht criterion), secondary market, denominated in the national '
             'currency.',
  'units': 'percent per annum',
  'variable': 'govt_10y'},
 {'countries': 'all 15',
  'dataset': 'FM',
  'meaning': 'Short-term interbank rate — 3-month EURIBOR, historical close, monthly average. '
             'One euro-area series applied to every euro-area country.',
  'units': 'percent per annum',
  'variable': 'short_rate'},
 {'countries': 'all 15',
  'dataset': 'MIR',
  'meaning': 'MFI interest rate on new business — loans to households for house purchase, '
             'annualised agreed rate, all maturities.',
  'units': 'percent per annum',
  'variable': 'lending_rate_hh'},
 {'countries': 'all 15',
  'dataset': 'MIR',
  'meaning': 'MFI interest rate on new business — loans to non-financial corporations, '
             'annualised agreed rate.',
  'units': 'percent per annum',
  'variable': 'lending_rate_nfc'},
 {'countries': 'all 15',
  'dataset': 'BSI',
  'meaning': 'MFI balance sheet item — outstanding amount of loans to households (incl. '
             'non-profit institutions serving households), all maturities. Stock, end of '
             'period; enters models as growth.',
  'units': 'EUR millions, outstanding amounts',
  'variable': 'credit_hh'},
 {'countries': 'all 15',
  'dataset': 'BSI',
  'meaning': 'MFI balance sheet item — outstanding amount of loans to non-financial '
             'corporations, all maturities. Stock, end of period; enters models as growth.',
  'units': 'EUR millions, outstanding amounts',
  'variable': 'credit_nfc'},
 {'countries': 'all 15',
  'dataset': 'AME',
  'meaning': 'AMECO (European Commission annual macro database, ECB mirror) real GDP at 2020 '
             'reference levels, item OVGD. National currency, so only growth rates are '
             'cross-country comparable. The last ~2 years of the series are EC forecasts (no '
             'in-band flag; cutover is rule-based).',
  'units': 'national currency, billions (2020 reference levels)',
  'variable': 'ameco_gdp'},
 {'countries': 'all 15',
  'dataset': 'AME',
  'meaning': 'AMECO unemployment rate, Eurostat definition, item ZUTN. The last ~2 years are '
             'EC forecasts (no in-band flag; cutover is rule-based).',
  'units': 'percent of active population',
  'variable': 'ameco_unemployment'},
 {'countries': 'all 15',
  'dataset': 'MNA',
  'meaning': 'Nominal gross domestic product at market prices, current prices (ESA 2010), '
             'seasonally and calendar adjusted.',
  'units': 'EUR millions, current prices',
  'variable': 'nominal_gdp'},
 {'countries': 'DK; GB; NO; PL; SE',
  'dataset': 'EXR',
  'meaning': 'ECB euro foreign-exchange reference rate — national currency per euro, monthly '
             'average. A rising value = depreciation against the euro.',
  'units': 'national currency per EUR',
  'variable': 'exchange_rate'},
 {'countries': 'all 15',
  'dataset': 'AME',
  'meaning': 'General government consolidated gross debt (Maastricht/EDP definition, ESA '
             '2010), AMECO annual series including ~2 years of European Commission forecasts.',
  'units': 'percent of GDP',
  'variable': 'gov_debt'},
 {'countries': 'all 15',
  'dataset': 'AME',
  'meaning': 'General government net lending (+) / net borrowing (−), EDP definition — the '
             'headline fiscal balance. AMECO annual series including EC forecasts.',
  'units': 'percent of GDP',
  'variable': 'gov_balance'},
 {'countries': 'AT; BE; DE; DK; ES; FR; GB; IE; IT; NL; PL; PT; SE',
  'dataset': 'CISS',
  'meaning': 'ECB Sovereign Systemic Stress Composite Indicator (CISS family) — measures '
             "stress in the country's sovereign bond market.",
  'units': 'index between 0 (calm) and 1 (extreme stress)',
  'variable': 'sovereign_stress'},
 {'countries': 'see section B',
  'dataset': 'MPD',
  'meaning': 'Real GDP growth, annual % change — Eurosystem/ECB staff projection '
             '(Macroeconomic Projection Database).',
  'units': 'percent',
  'variable': 'MPD YER'},
 {'countries': 'see section B',
  'dataset': 'MPD',
  'meaning': 'HICP inflation, annual % change — Eurosystem/ECB staff projection.',
  'units': 'percent',
  'variable': 'MPD HIC'},
 {'countries': 'see section B',
  'dataset': 'MPD',
  'meaning': 'Unemployment rate — Eurosystem/ECB staff projection.',
  'units': 'percent of labour force',
  'variable': 'MPD URX'},
 {'countries': 'see section B',
  'dataset': 'AME',
  'meaning': 'Real GDP, 2020 reference levels — AMECO history plus ~2 years of EC forecast.',
  'units': 'national currency, billions',
  'variable': 'AME OVGD'},
 {'countries': 'see section B',
  'dataset': 'AME',
  'meaning': 'Unemployment rate — AMECO history plus EC forecast.',
  'units': 'percent of active population',
  'variable': 'AME ZUTN'},
 {'countries': 'see section B',
  'dataset': 'AME',
  'meaning': 'Government gross debt (EDP) — AMECO history plus EC forecast.',
  'units': 'percent of GDP',
  'variable': 'AME UDGGL'},
 {'countries': 'see section B',
  'dataset': 'AME',
  'meaning': 'Government net lending/borrowing (EDP) — AMECO history plus EC forecast.',
  'units': 'percent of GDP',
  'variable': 'AME UBLGE'}])
show(VARIABLE_GUIDE, "Variable dictionary — labels, meaning, units, coverage")

,countries,dataset,meaning,units,variable
0,all 15,MNA,"Real gross domestic product, chain-linked volume (ESA 2010, reference year 2020), seasonally and calendar adjusted. National accounts main aggregate; enters models as growth (T4/T6).","EUR millions, chain-linked volume",gdp_real
1,all 15,ICP,"HICP inflation — Harmonised Index of Consumer Prices, all items, annual rate of change. The EU-harmonised consumer inflation measure.",percent per annum (annual rate of change),hicp
2,all 15,LFSI,"Unemployment rate, ILO definition, ages 15-74, total, seasonally adjusted (Eurostat Labour Force Survey via ECB LFSI).",percent of labour force,unemployment
3,all 15,RPP,"Residential property prices, all types of dwellings (new and existing), whole country, index. ECB RPP indicator compiled from national/BIS sources (source code 4).",index,house_prices
4,all 15,IRS,"10-year government bond yield — long-term interest rate for EU convergence purposes (Maastricht criterion), secondary market, denominated in the national currency.",percent per annum,govt_10y
5,all 15,FM,"Short-term interbank rate — 3-month EURIBOR, historical close, monthly average. One euro-area series applied to every euro-area country.",percent per annum,short_rate
6,all 15,MIR,"MFI interest rate on new business — loans to households for house purchase, annualised agreed rate, all maturities.",percent per annum,lending_rate_hh
7,all 15,MIR,"MFI interest rate on new business — loans to non-financial corporations, annualised agreed rate.",percent per annum,lending_rate_nfc
8,all 15,BSI,"MFI balance sheet item — outstanding amount of loans to households (incl. non-profit institutions serving households), all maturities. Stock, end of period; enters models as growth.","EUR millions, outstanding amounts",credit_hh
9,all 15,BSI,"MFI balance sheet item — outstanding amount of loans to non-financial corporations, all maturities. Stock, end of period; enters models as growth.","EUR millions, outstanding amounts",credit_nfc


---
## B — Series labels (no data yet)

A series is addressed by a dot-separated **key** whose positions are the
dataset's dimensions — e.g. `ICP.M.DE.N.000000.4.ANR` is *monthly, Germany,
neither seasonally adjusted, all-items HICP, annual rate of change*.

Below we take one **key pattern per variable**, expand it for each variable's
country scope, and check every candidate against the official series
catalogues from section A — keeping only the labels that really exist.


In [3]:
KEY_PATTERNS = {   # variable -> (dataset, key pattern)
    'gdp_real': ('MNA', 'Q.Y.{COUNTRY}.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N'),
    'hicp': ('ICP', 'M.{COUNTRY}.N.000000.4.ANR'),
    'unemployment': ('LFSI', 'M.{COUNTRY}.S.UNEHRT.TOTAL0.15_74.T'),
    'house_prices': ('RPP', 'Q.{COUNTRY}.N.TD.00.4.00'),
    'govt_10y': ('IRS', 'M.{COUNTRY}.L.L40.CI.0000.{CURRENCY}.N.Z'),
    'short_rate': ('FM', 'M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA'),
    'lending_rate_hh': ('MIR', 'M.{COUNTRY}.B.A2C.AM.R.A.2250.EUR.N'),
    'lending_rate_nfc': ('MIR', 'M.{COUNTRY}.B.A2A.A.R.A.2240.EUR.N'),
    'credit_hh': ('BSI', 'M.{COUNTRY}.N.A.A20.A.1.U2.2250.Z01.E'),
    'credit_nfc': ('BSI', 'M.{COUNTRY}.N.A.A20.A.1.U2.2240.Z01.E'),
    'ameco_gdp': ('AME', 'A.{CC3}.1.0.0.0.OVGD'),
    'ameco_unemployment': ('AME', 'A.{CC3}.1.0.0.0.ZUTN'),
    'nominal_gdp': ('MNA', 'Q.Y.{COUNTRY}.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N'),
    'exchange_rate': ('EXR', 'M.{CURRENCY}.EUR.SP00.A'),
    'gov_debt': ('AME', 'A.{CC3}.1.0.319.0.UDGGL'),
    'gov_balance': ('AME', 'A.{CC3}.1.0.319.0.UBLGE'),
    'sovereign_stress': ('CISS', 'M.{COUNTRY}.Z0Z.4F.EC.SOV_CI.IDX'),
}
SCOPE = {          # variables not available/meaningful for all countries
    'exchange_rate': ['DK', 'GB', 'NO', 'PL', 'SE'],
    'sovereign_stress': ['AT', 'BE', 'DE', 'DK', 'ES', 'FR', 'GB', 'IE', 'IT', 'NL', 'PL', 'PT', 'SE'],
}
CURRENCY = {'AT': 'EUR',
 'BE': 'EUR',
 'DE': 'EUR',
 'DK': 'DKK',
 'ES': 'EUR',
 'FR': 'EUR',
 'GB': 'GBP',
 'IE': 'EUR',
 'IT': 'EUR',
 'LU': 'EUR',
 'NL': 'EUR',
 'NO': 'NOK',
 'PL': 'PLN',
 'PT': 'EUR',
 'SE': 'SEK'}   # EXR/IRS keys use the national currency
CC3 = {'AT': 'AUT',
 'BE': 'BEL',
 'DE': 'DEU',
 'DK': 'DNK',
 'ES': 'ESP',
 'FR': 'FRA',
 'GB': 'GBR',
 'IE': 'IRL',
 'IT': 'ITA',
 'LU': 'LUX',
 'NL': 'NLD',
 'NO': 'NOR',
 'PL': 'POL',
 'PT': 'PRT',
 'SE': 'SWE'}      # AMECO uses 3-letter codes

def expand(pattern, cc):
    return (pattern.replace("{COUNTRY}", cc)
                   .replace("{CURRENCY}", CURRENCY[cc])
                   .replace("{CC3}", CC3[cc]))

candidates = {var: {cc: f"{flow}.{expand(tpl, cc)}"
                    for cc in SCOPE.get(var, COUNTRIES)}
              for var, (flow, tpl) in KEY_PATTERNS.items()}
print(f"{sum(len(v) for v in candidates.values())} candidate labels "
      f"({len(candidates)} variables)")

VARIABLE_GUIDE.head(20)

243 candidate labels (17 variables)


,countries,dataset,meaning,units,variable
0,all 15,MNA,"Real gross domestic product, chain-linked volume (ESA 2010, reference year 2020), seasonally and calendar ...","EUR millions, chain-linked volume",gdp_real
1,all 15,ICP,"HICP inflation — Harmonised Index of Consumer Prices, all items, annual rate of change. The EU-harmonised ...",percent per annum (annual rate of change),hicp
2,all 15,LFSI,"Unemployment rate, ILO definition, ages 15-74, total, seasonally adjusted (Eurostat Labour Force Survey vi...",percent of labour force,unemployment
3,all 15,RPP,"Residential property prices, all types of dwellings (new and existing), whole country, index. ECB RPP indi...",index,house_prices
4,all 15,IRS,10-year government bond yield — long-term interest rate for EU convergence purposes (Maastricht criterion)...,percent per annum,govt_10y
5,all 15,FM,"Short-term interbank rate — 3-month EURIBOR, historical close, monthly average. One euro-area series appli...",percent per annum,short_rate
6,all 15,MIR,"MFI interest rate on new business — loans to households for house purchase, annualised agreed rate, all ma...",percent per annum,lending_rate_hh
7,all 15,MIR,"MFI interest rate on new business — loans to non-financial corporations, annualised agreed rate.",percent per annum,lending_rate_nfc
8,all 15,BSI,MFI balance sheet item — outstanding amount of loans to households (incl. non-profit institutions serving ...,"EUR millions, outstanding amounts",credit_hh
9,all 15,BSI,"MFI balance sheet item — outstanding amount of loans to non-financial corporations, all maturities. Stock,...","EUR millions, outstanding amounts",credit_nfc


In [4]:
def fetch_catalogue(flow):
    """Download and parse a dataset's official series catalogue (zipped CSV).
    A handful of ICP rows carry stray quotes that split the title field —
    repaired by rejoining so the trailing dimension columns stay aligned."""
    resp = SESSION.get(f"{PORTAL}/data/datasets/{flow}/download", timeout=120)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        with zf.open(zf.namelist()[0]) as fh:
            rows = list(csv.reader(io.TextIOWrapper(fh, encoding="utf-8",
                                                    errors="replace")))
    header, body, n, title_idx = rows[0], rows[1:], len(rows[0]), 13
    clean = []
    for r in body:
        if len(r) > n:
            extra = len(r) - n
            r = (r[:title_idx] + [",".join(r[title_idx:title_idx + extra + 1])]
                 + r[title_idx + extra + 1:])
        elif len(r) < n:
            r += [""] * (n - len(r))
        clean.append(r)
    return pd.DataFrame(clean, columns=header)

flows = sorted({flow for flow, _ in KEY_PATTERNS.values()})
catalogues = {}
for flow in flows:
    catalogues[flow] = fetch_catalogue(flow)
    print(f"{flow:5} catalogue: {len(catalogues[flow]):>7,} series")

AME   catalogue:     124 series
BSI   catalogue:  66,785 series
CISS  catalogue:      60 series
EXR   catalogue:   4,027 series
FM    catalogue:     115 series
ICP   catalogue:  60,691 series
IRS   catalogue:      40 series
LFSI  catalogue:   1,212 series
MIR   catalogue:   8,127 series
MNA   catalogue:  29,593 series
RPP   catalogue:     228 series


BSI   catalogue:  66,785 series


CISS  catalogue:      60 series


EXR   catalogue:   4,027 series


FM    catalogue:     115 series


...

IRS   catalogue:      40 series


LFSI  catalogue:   1,212 series


MIR   catalogue:   8,127 series


MNA   catalogue:  29,593 series


RPP   catalogue:     228 series


In [5]:
rows = []
for var, (flow, _) in KEY_PATTERNS.items():
    cat = catalogues[flow].set_index("series key")
    title_col = "title_compl" if "title_compl" in cat.columns else "title"
    for cc, key in candidates[var].items():
        hit = key in cat.index
        rows.append({"variable": var, "country": cc, "series_key": key,
                     "exists": hit,
                     "official_title": str(cat.loc[key, title_col])[:110] if hit else ""})
labels = pd.DataFrame(rows)
print(f"labels confirmed in the official catalogues: "
      f"{int(labels.exists.sum())}/{len(labels)}")
print()
missing = labels[~labels.exists].groupby("variable")["country"].apply(", ".join)
show(missing.to_frame("not available for"),
     "Missing labels — structural gaps, not errors")

labels confirmed in the official catalogues: 217/243



,not available for
variable,
ameco_gdp,NO
ameco_unemployment,NO
credit_hh,"DK, NO"
credit_nfc,"DK, NO"
gdp_real,NO
gov_balance,NO
gov_debt,NO
govt_10y,NO
hicp,NO


Worth pausing on:

- **Norway** appears only in the exchange-rate dataset — it is neither an EU
  member nor an ECB reporter, so activity/prices/credit need a national
  source. That is a real finding about ECB coverage, not a query mistake.
- The GB unemployment series (LFSI) was **discontinued** after Brexit;
  AMECO's `ZUTN` fills that role in the forecast block.
- Bank lending rates (`MIR`) and 3M EURIBOR exist for euro-area countries
  only; sovereign-stress indicators cover 13 of our 15.

### The pinned label list

The cell below **hardcodes the result of the extraction above** — the full
confirmed inventory plus the two forecast blocks — so the rest of the
notebook (and anyone copying just sections C-D) does not depend on the
catalogue downloads.

- `MPD` keys pin the **December 2025 projection round** (code `A25`; letters:
  `A` = December, `G` = June — the only rounds with country detail). A round
  can exist *structurally* before its values are published — the June 2026
  round was empty at the time of writing, which is why we pin `A25`. Its
  values run to 2028.
- `AME` keys return history **and** ~2 years of EC forecasts in one series
  (no in-band flag — treat years beyond the last completed year as forecast).


In [6]:
# Pinned output of section B — catalogue-validated series labels
HISTORY_SERIES = {'ameco_gdp': {'AT': 'AME.A.AUT.1.0.0.0.OVGD',
               'BE': 'AME.A.BEL.1.0.0.0.OVGD',
               'DE': 'AME.A.DEU.1.0.0.0.OVGD',
               'DK': 'AME.A.DNK.1.0.0.0.OVGD',
               'ES': 'AME.A.ESP.1.0.0.0.OVGD',
               'FR': 'AME.A.FRA.1.0.0.0.OVGD',
               'GB': 'AME.A.GBR.1.0.0.0.OVGD',
               'IE': 'AME.A.IRL.1.0.0.0.OVGD',
               'IT': 'AME.A.ITA.1.0.0.0.OVGD',
               'LU': 'AME.A.LUX.1.0.0.0.OVGD',
               'NL': 'AME.A.NLD.1.0.0.0.OVGD',
               'PL': 'AME.A.POL.1.0.0.0.OVGD',
               'PT': 'AME.A.PRT.1.0.0.0.OVGD',
               'SE': 'AME.A.SWE.1.0.0.0.OVGD'},
 'ameco_unemployment': {'AT': 'AME.A.AUT.1.0.0.0.ZUTN',
                        'BE': 'AME.A.BEL.1.0.0.0.ZUTN',
                        'DE': 'AME.A.DEU.1.0.0.0.ZUTN',
                        'DK': 'AME.A.DNK.1.0.0.0.ZUTN',
                        'ES': 'AME.A.ESP.1.0.0.0.ZUTN',
                        'FR': 'AME.A.FRA.1.0.0.0.ZUTN',
                        'GB': 'AME.A.GBR.1.0.0.0.ZUTN',
                        'IE': 'AME.A.IRL.1.0.0.0.ZUTN',
                        'IT': 'AME.A.ITA.1.0.0.0.ZUTN',
                        'LU': 'AME.A.LUX.1.0.0.0.ZUTN',
                        'NL': 'AME.A.NLD.1.0.0.0.ZUTN',
                        'PL': 'AME.A.POL.1.0.0.0.ZUTN',
                        'PT': 'AME.A.PRT.1.0.0.0.ZUTN',
                        'SE': 'AME.A.SWE.1.0.0.0.ZUTN'},
 'credit_hh': {'AT': 'BSI.M.AT.N.A.A20.A.1.U2.2250.Z01.E',
               'BE': 'BSI.M.BE.N.A.A20.A.1.U2.2250.Z01.E',
               'DE': 'BSI.M.DE.N.A.A20.A.1.U2.2250.Z01.E',
               'ES': 'BSI.M.ES.N.A.A20.A.1.U2.2250.Z01.E',
               'FR': 'BSI.M.FR.N.A.A20.A.1.U2.2250.Z01.E',
               'GB': 'BSI.M.GB.N.A.A20.A.1.U2.2250.Z01.E',
               'IE': 'BSI.M.IE.N.A.A20.A.1.U2.2250.Z01.E',
               'IT': 'BSI.M.IT.N.A.A20.A.1.U2.2250.Z01.E',
               'LU': 'BSI.M.LU.N.A.A20.A.1.U2.2250.Z01.E',
               'NL': 'BSI.M.NL.N.A.A20.A.1.U2.2250.Z01.E',
               'PL': 'BSI.M.PL.N.A.A20.A.1.U2.2250.Z01.E',
               'PT': 'BSI.M.PT.N.A.A20.A.1.U2.2250.Z01.E',
               'SE': 'BSI.M.SE.N.A.A20.A.1.U2.2250.Z01.E'},
 'credit_nfc': {'AT': 'BSI.M.AT.N.A.A20.A.1.U2.2240.Z01.E',
                'BE': 'BSI.M.BE.N.A.A20.A.1.U2.2240.Z01.E',
                'DE': 'BSI.M.DE.N.A.A20.A.1.U2.2240.Z01.E',
                'ES': 'BSI.M.ES.N.A.A20.A.1.U2.2240.Z01.E',
                'FR': 'BSI.M.FR.N.A.A20.A.1.U2.2240.Z01.E',
                'GB': 'BSI.M.GB.N.A.A20.A.1.U2.2240.Z01.E',
                'IE': 'BSI.M.IE.N.A.A20.A.1.U2.2240.Z01.E',
                'IT': 'BSI.M.IT.N.A.A20.A.1.U2.2240.Z01.E',
                'LU': 'BSI.M.LU.N.A.A20.A.1.U2.2240.Z01.E',
                'NL': 'BSI.M.NL.N.A.A20.A.1.U2.2240.Z01.E',
                'PL': 'BSI.M.PL.N.A.A20.A.1.U2.2240.Z01.E',
                'PT': 'BSI.M.PT.N.A.A20.A.1.U2.2240.Z01.E',
                'SE': 'BSI.M.SE.N.A.A20.A.1.U2.2240.Z01.E'},
 'exchange_rate': {'DK': 'EXR.M.DKK.EUR.SP00.A',
                   'GB': 'EXR.M.GBP.EUR.SP00.A',
                   'NO': 'EXR.M.NOK.EUR.SP00.A',
                   'PL': 'EXR.M.PLN.EUR.SP00.A',
                   'SE': 'EXR.M.SEK.EUR.SP00.A'},
 'gdp_real': {'AT': 'MNA.Q.Y.AT.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'BE': 'MNA.Q.Y.BE.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'DE': 'MNA.Q.Y.DE.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'DK': 'MNA.Q.Y.DK.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'ES': 'MNA.Q.Y.ES.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'FR': 'MNA.Q.Y.FR.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'GB': 'MNA.Q.Y.GB.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'IE': 'MNA.Q.Y.IE.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'IT': 'MNA.Q.Y.IT.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'LU': 'MNA.Q.Y.LU.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'NL': 'MNA.Q.Y.NL.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'PL': 'MNA.Q.Y.PL.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'PT': 'MNA.Q.Y.PT.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N',
              'SE': 'MNA.Q.Y.SE.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N'},
 'gov_balance': {'AT': 'AME.A.AUT.1.0.319.0.UBLGE',
                 'BE': 'AME.A.BEL.1.0.319.0.UBLGE',
                 'DE': 'AME.A.DEU.1.0.319.0.UBLGE',
                 'DK': 'AME.A.DNK.1.0.319.0.UBLGE',
                 'ES': 'AME.A.ESP.1.0.319.0.UBLGE',
                 'FR': 'AME.A.FRA.1.0.319.0.UBLGE',
                 'GB': 'AME.A.GBR.1.0.319.0.UBLGE',
                 'IE': 'AME.A.IRL.1.0.319.0.UBLGE',
                 'IT': 'AME.A.ITA.1.0.319.0.UBLGE',
                 'LU': 'AME.A.LUX.1.0.319.0.UBLGE',
                 'NL': 'AME.A.NLD.1.0.319.0.UBLGE',
                 'PL': 'AME.A.POL.1.0.319.0.UBLGE',
                 'PT': 'AME.A.PRT.1.0.319.0.UBLGE',
                 'SE': 'AME.A.SWE.1.0.319.0.UBLGE'},
 'gov_debt': {'AT': 'AME.A.AUT.1.0.319.0.UDGGL',
              'BE': 'AME.A.BEL.1.0.319.0.UDGGL',
              'DE': 'AME.A.DEU.1.0.319.0.UDGGL',
              'DK': 'AME.A.DNK.1.0.319.0.UDGGL',
              'ES': 'AME.A.ESP.1.0.319.0.UDGGL',
              'FR': 'AME.A.FRA.1.0.319.0.UDGGL',
              'GB': 'AME.A.GBR.1.0.319.0.UDGGL',
              'IE': 'AME.A.IRL.1.0.319.0.UDGGL',
              'IT': 'AME.A.ITA.1.0.319.0.UDGGL',
              'LU': 'AME.A.LUX.1.0.319.0.UDGGL',
              'NL': 'AME.A.NLD.1.0.319.0.UDGGL',
              'PL': 'AME.A.POL.1.0.319.0.UDGGL',
              'PT': 'AME.A.PRT.1.0.319.0.UDGGL',
              'SE': 'AME.A.SWE.1.0.319.0.UDGGL'},
 'govt_10y': {'AT': 'IRS.M.AT.L.L40.CI.0000.EUR.N.Z',
              'BE': 'IRS.M.BE.L.L40.CI.0000.EUR.N.Z',
              'DE': 'IRS.M.DE.L.L40.CI.0000.EUR.N.Z',
              'DK': 'IRS.M.DK.L.L40.CI.0000.DKK.N.Z',
              'ES': 'IRS.M.ES.L.L40.CI.0000.EUR.N.Z',
              'FR': 'IRS.M.FR.L.L40.CI.0000.EUR.N.Z',
              'GB': 'IRS.M.GB.L.L40.CI.0000.GBP.N.Z',
              'IE': 'IRS.M.IE.L.L40.CI.0000.EUR.N.Z',
              'IT': 'IRS.M.IT.L.L40.CI.0000.EUR.N.Z',
              'LU': 'IRS.M.LU.L.L40.CI.0000.EUR.N.Z',
              'NL': 'IRS.M.NL.L.L40.CI.0000.EUR.N.Z',
              'PL': 'IRS.M.PL.L.L40.CI.0000.PLN.N.Z',
              'PT': 'IRS.M.PT.L.L40.CI.0000.EUR.N.Z',
              'SE': 'IRS.M.SE.L.L40.CI.0000.SEK.N.Z'},
 'hicp': {'AT': 'ICP.M.AT.N.000000.4.ANR',
          'BE': 'ICP.M.BE.N.000000.4.ANR',
          'DE': 'ICP.M.DE.N.000000.4.ANR',
          'DK': 'ICP.M.DK.N.000000.4.ANR',
          'ES': 'ICP.M.ES.N.000000.4.ANR',
          'FR': 'ICP.M.FR.N.000000.4.ANR',
          'GB': 'ICP.M.GB.N.000000.4.ANR',
          'IE': 'ICP.M.IE.N.000000.4.ANR',
          'IT': 'ICP.M.IT.N.000000.4.ANR',
          'LU': 'ICP.M.LU.N.000000.4.ANR',
          'NL': 'ICP.M.NL.N.000000.4.ANR',
          'PL': 'ICP.M.PL.N.000000.4.ANR',
          'PT': 'ICP.M.PT.N.000000.4.ANR',
          'SE': 'ICP.M.SE.N.000000.4.ANR'},
 'house_prices': {'AT': 'RPP.Q.AT.N.TD.00.4.00',
                  'BE': 'RPP.Q.BE.N.TD.00.4.00',
                  'DE': 'RPP.Q.DE.N.TD.00.4.00',
                  'DK': 'RPP.Q.DK.N.TD.00.4.00',
                  'ES': 'RPP.Q.ES.N.TD.00.4.00',
                  'FR': 'RPP.Q.FR.N.TD.00.4.00',
                  'GB': 'RPP.Q.GB.N.TD.00.4.00',
                  'IE': 'RPP.Q.IE.N.TD.00.4.00',
                  'IT': 'RPP.Q.IT.N.TD.00.4.00',
                  'LU': 'RPP.Q.LU.N.TD.00.4.00',
                  'NL': 'RPP.Q.NL.N.TD.00.4.00',
                  'PL': 'RPP.Q.PL.N.TD.00.4.00',
                  'PT': 'RPP.Q.PT.N.TD.00.4.00',
                  'SE': 'RPP.Q.SE.N.TD.00.4.00'},
 'lending_rate_hh': {'AT': 'MIR.M.AT.B.A2C.AM.R.A.2250.EUR.N',
                     'BE': 'MIR.M.BE.B.A2C.AM.R.A.2250.EUR.N',
                     'DE': 'MIR.M.DE.B.A2C.AM.R.A.2250.EUR.N',
                     'ES': 'MIR.M.ES.B.A2C.AM.R.A.2250.EUR.N',
                     'FR': 'MIR.M.FR.B.A2C.AM.R.A.2250.EUR.N',
                     'IE': 'MIR.M.IE.B.A2C.AM.R.A.2250.EUR.N',
                     'IT': 'MIR.M.IT.B.A2C.AM.R.A.2250.EUR.N',
                     'LU': 'MIR.M.LU.B.A2C.AM.R.A.2250.EUR.N',
                     'NL': 'MIR.M.NL.B.A2C.AM.R.A.2250.EUR.N',
                     'PT': 'MIR.M.PT.B.A2C.AM.R.A.2250.EUR.N'},
 'lending_rate_nfc': {'AT': 'MIR.M.AT.B.A2A.A.R.A.2240.EUR.N',
                      'BE': 'MIR.M.BE.B.A2A.A.R.A.2240.EUR.N',
                      'DE': 'MIR.M.DE.B.A2A.A.R.A.2240.EUR.N',
                      'ES': 'MIR.M.ES.B.A2A.A.R.A.2240.EUR.N',
                      'FR': 'MIR.M.FR.B.A2A.A.R.A.2240.EUR.N',
                      'IE': 'MIR.M.IE.B.A2A.A.R.A.2240.EUR.N',
                      'IT': 'MIR.M.IT.B.A2A.A.R.A.2240.EUR.N',
                      'LU': 'MIR.M.LU.B.A2A.A.R.A.2240.EUR.N',
                      'NL': 'MIR.M.NL.B.A2A.A.R.A.2240.EUR.N',
                      'PT': 'MIR.M.PT.B.A2A.A.R.A.2240.EUR.N'},
 'nominal_gdp': {'AT': 'MNA.Q.Y.AT.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'BE': 'MNA.Q.Y.BE.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'DE': 'MNA.Q.Y.DE.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'DK': 'MNA.Q.Y.DK.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'ES': 'MNA.Q.Y.ES.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'FR': 'MNA.Q.Y.FR.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'IE': 'MNA.Q.Y.IE.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'IT': 'MNA.Q.Y.IT.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'LU': 'MNA.Q.Y.LU.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'NL': 'MNA.Q.Y.NL.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'PL': 'MNA.Q.Y.PL.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'PT': 'MNA.Q.Y.PT.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N',
                 'SE': 'MNA.Q.Y.SE.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.V.N'},
 'short_rate': {'AT': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'BE': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'DE': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'DK': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'ES': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'FR': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'GB': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'IE': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'IT': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'LU': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'NL': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'NO': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'PL': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'PT': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA',
                'SE': 'FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA'},
 'sovereign_stress': {'AT': 'CISS.M.AT.Z0Z.4F.EC.SOV_CI.IDX',
                      'BE': 'CISS.M.BE.Z0Z.4F.EC.SOV_CI.IDX',
                      'DE': 'CISS.M.DE.Z0Z.4F.EC.SOV_CI.IDX',
                      'DK': 'CISS.M.DK.Z0Z.4F.EC.SOV_CI.IDX',
                      'ES': 'CISS.M.ES.Z0Z.4F.EC.SOV_CI.IDX',
                      'FR': 'CISS.M.FR.Z0Z.4F.EC.SOV_CI.IDX',
                      'GB': 'CISS.M.GB.Z0Z.4F.EC.SOV_CI.IDX',
                      'IE': 'CISS.M.IE.Z0Z.4F.EC.SOV_CI.IDX',
                      'IT': 'CISS.M.IT.Z0Z.4F.EC.SOV_CI.IDX',
                      'NL': 'CISS.M.NL.Z0Z.4F.EC.SOV_CI.IDX',
                      'PL': 'CISS.M.PL.Z0Z.4F.EC.SOV_CI.IDX',
                      'PT': 'CISS.M.PT.Z0Z.4F.EC.SOV_CI.IDX',
                      'SE': 'CISS.M.SE.Z0Z.4F.EC.SOV_CI.IDX'},
 'unemployment': {'AT': 'LFSI.M.AT.S.UNEHRT.TOTAL0.15_74.T',
                  'BE': 'LFSI.M.BE.S.UNEHRT.TOTAL0.15_74.T',
                  'DE': 'LFSI.M.DE.S.UNEHRT.TOTAL0.15_74.T',
                  'DK': 'LFSI.M.DK.S.UNEHRT.TOTAL0.15_74.T',
                  'ES': 'LFSI.M.ES.S.UNEHRT.TOTAL0.15_74.T',
                  'FR': 'LFSI.M.FR.S.UNEHRT.TOTAL0.15_74.T',
                  'IE': 'LFSI.M.IE.S.UNEHRT.TOTAL0.15_74.T',
                  'IT': 'LFSI.M.IT.S.UNEHRT.TOTAL0.15_74.T',
                  'LU': 'LFSI.M.LU.S.UNEHRT.TOTAL0.15_74.T',
                  'NL': 'LFSI.M.NL.S.UNEHRT.TOTAL0.15_74.T',
                  'PL': 'LFSI.M.PL.S.UNEHRT.TOTAL0.15_74.T',
                  'PT': 'LFSI.M.PT.S.UNEHRT.TOTAL0.15_74.T',
                  'SE': 'LFSI.M.SE.S.UNEHRT.TOTAL0.15_74.T'}}

MPD_ROUND = "2025-12"  # December 2025 Eurosystem projections (code A25)
MPD_SERIES = {'HIC': {'AT': 'MPD.A.AT.HIC.A.A25.0000',
         'BE': 'MPD.A.BE.HIC.A.A25.0000',
         'DE': 'MPD.A.DE.HIC.A.A25.0000',
         'ES': 'MPD.A.ES.HIC.A.A25.0000',
         'FR': 'MPD.A.FR.HIC.A.A25.0000',
         'IE': 'MPD.A.IE.HIC.A.A25.0000',
         'IT': 'MPD.A.IT.HIC.A.A25.0000',
         'LU': 'MPD.A.LU.HIC.A.A25.0000',
         'NL': 'MPD.A.NL.HIC.A.A25.0000',
         'PT': 'MPD.A.PT.HIC.A.A25.0000'},
 'URX': {'AT': 'MPD.A.AT.URX.F.A25.0000',
         'BE': 'MPD.A.BE.URX.F.A25.0000',
         'DE': 'MPD.A.DE.URX.F.A25.0000',
         'ES': 'MPD.A.ES.URX.F.A25.0000',
         'FR': 'MPD.A.FR.URX.F.A25.0000',
         'IE': 'MPD.A.IE.URX.F.A25.0000',
         'IT': 'MPD.A.IT.URX.F.A25.0000',
         'LU': 'MPD.A.LU.URX.F.A25.0000',
         'NL': 'MPD.A.NL.URX.F.A25.0000',
         'PT': 'MPD.A.PT.URX.F.A25.0000'},
 'YER': {'AT': 'MPD.A.AT.YER.A.A25.0000',
         'BE': 'MPD.A.BE.YER.A.A25.0000',
         'DE': 'MPD.A.DE.YER.A.A25.0000',
         'ES': 'MPD.A.ES.YER.A.A25.0000',
         'FR': 'MPD.A.FR.YER.A.A25.0000',
         'IE': 'MPD.A.IE.YER.A.A25.0000',
         'IT': 'MPD.A.IT.YER.A.A25.0000',
         'LU': 'MPD.A.LU.YER.A.A25.0000',
         'NL': 'MPD.A.NL.YER.A.A25.0000',
         'PT': 'MPD.A.PT.YER.A.A25.0000'}}

AME_SERIES = {'OVGD': {'AT': 'AME.A.AUT.1.0.0.0.OVGD',
          'BE': 'AME.A.BEL.1.0.0.0.OVGD',
          'DE': 'AME.A.DEU.1.0.0.0.OVGD',
          'DK': 'AME.A.DNK.1.0.0.0.OVGD',
          'ES': 'AME.A.ESP.1.0.0.0.OVGD',
          'FR': 'AME.A.FRA.1.0.0.0.OVGD',
          'GB': 'AME.A.GBR.1.0.0.0.OVGD',
          'IE': 'AME.A.IRL.1.0.0.0.OVGD',
          'IT': 'AME.A.ITA.1.0.0.0.OVGD',
          'LU': 'AME.A.LUX.1.0.0.0.OVGD',
          'NL': 'AME.A.NLD.1.0.0.0.OVGD',
          'PL': 'AME.A.POL.1.0.0.0.OVGD',
          'PT': 'AME.A.PRT.1.0.0.0.OVGD',
          'SE': 'AME.A.SWE.1.0.0.0.OVGD'},
 'UBLGE': {'AT': 'AME.A.AUT.1.0.319.0.UBLGE',
           'BE': 'AME.A.BEL.1.0.319.0.UBLGE',
           'DE': 'AME.A.DEU.1.0.319.0.UBLGE',
           'DK': 'AME.A.DNK.1.0.319.0.UBLGE',
           'ES': 'AME.A.ESP.1.0.319.0.UBLGE',
           'FR': 'AME.A.FRA.1.0.319.0.UBLGE',
           'GB': 'AME.A.GBR.1.0.319.0.UBLGE',
           'IE': 'AME.A.IRL.1.0.319.0.UBLGE',
           'IT': 'AME.A.ITA.1.0.319.0.UBLGE',
           'LU': 'AME.A.LUX.1.0.319.0.UBLGE',
           'NL': 'AME.A.NLD.1.0.319.0.UBLGE',
           'PL': 'AME.A.POL.1.0.319.0.UBLGE',
           'PT': 'AME.A.PRT.1.0.319.0.UBLGE',
           'SE': 'AME.A.SWE.1.0.319.0.UBLGE'},
 'UDGGL': {'AT': 'AME.A.AUT.1.0.319.0.UDGGL',
           'BE': 'AME.A.BEL.1.0.319.0.UDGGL',
           'DE': 'AME.A.DEU.1.0.319.0.UDGGL',
           'DK': 'AME.A.DNK.1.0.319.0.UDGGL',
           'ES': 'AME.A.ESP.1.0.319.0.UDGGL',
           'FR': 'AME.A.FRA.1.0.319.0.UDGGL',
           'GB': 'AME.A.GBR.1.0.319.0.UDGGL',
           'IE': 'AME.A.IRL.1.0.319.0.UDGGL',
           'IT': 'AME.A.ITA.1.0.319.0.UDGGL',
           'LU': 'AME.A.LUX.1.0.319.0.UDGGL',
           'NL': 'AME.A.NLD.1.0.319.0.UDGGL',
           'PL': 'AME.A.POL.1.0.319.0.UDGGL',
           'PT': 'AME.A.PRT.1.0.319.0.UDGGL',
           'SE': 'AME.A.SWE.1.0.319.0.UDGGL'},
 'ZUTN': {'AT': 'AME.A.AUT.1.0.0.0.ZUTN',
          'BE': 'AME.A.BEL.1.0.0.0.ZUTN',
          'DE': 'AME.A.DEU.1.0.0.0.ZUTN',
          'DK': 'AME.A.DNK.1.0.0.0.ZUTN',
          'ES': 'AME.A.ESP.1.0.0.0.ZUTN',
          'FR': 'AME.A.FRA.1.0.0.0.ZUTN',
          'GB': 'AME.A.GBR.1.0.0.0.ZUTN',
          'IE': 'AME.A.IRL.1.0.0.0.ZUTN',
          'IT': 'AME.A.ITA.1.0.0.0.ZUTN',
          'LU': 'AME.A.LUX.1.0.0.0.ZUTN',
          'NL': 'AME.A.NLD.1.0.0.0.ZUTN',
          'PL': 'AME.A.POL.1.0.0.0.ZUTN',
          'PT': 'AME.A.PRT.1.0.0.0.ZUTN',
          'SE': 'AME.A.SWE.1.0.0.0.ZUTN'}}

n_hist = sum(len(v) for v in HISTORY_SERIES.values())
n_fc = sum(len(v) for v in MPD_SERIES.values()) + sum(len(v) for v in AME_SERIES.values())
print(f"pinned labels: {n_hist} history + {n_fc} forecast = {n_hist + n_fc} series")

pinned labels: 217 history + 86 forecast = 303 series


---
## C — Downloading the data

One GET per series:
`https://data-api.ecb.europa.eu/service/data/{DATASET}/{KEY}?format=csvdata`
returns a tidy CSV with `TIME_PERIOD` and `OBS_VALUE` columns. Each response
is parsed into a pandas Series indexed by `pandas.Period` (annual, quarterly
or monthly — inferred from the `TIME_PERIOD` format). The fetcher retries
with backoff: long request sequences occasionally hit a dropped keep-alive
connection, which is transient.


In [7]:
def to_period(tp):
    tp = str(tp)
    if re.fullmatch(r"\d{4}", tp):        return pd.Period(tp, freq="Y")
    if re.fullmatch(r"\d{4}-Q[1-4]", tp): return pd.Period(tp, freq="Q")
    if re.fullmatch(r"\d{4}-\d{2}", tp):  return pd.Period(tp, freq="M")
    raise ValueError(tp)

def fetch_series(full_key, retries=3):
    """full_key = 'DATASET.rest.of.key' -> pandas Series (None if no data)."""
    flow, key = full_key.split(".", 1)
    last_err = None
    for attempt in range(retries):
        try:
            resp = SESSION.get(f"{API}/{flow}/{key}",
                               params={"format": "csvdata"}, timeout=120)
            if resp.status_code == 404 or not resp.text.strip():
                return None
            resp.raise_for_status()
            break
        except (requests.ConnectionError, requests.Timeout,
                requests.HTTPError) as err:
            last_err = err
            time.sleep(1.5 * (attempt + 1))
    else:
        raise RuntimeError(f"{full_key}: failed after {retries} attempts: {last_err}")
    df = pd.read_csv(io.StringIO(resp.text))
    if "OBS_VALUE" not in df.columns or df.empty:
        return None
    s = pd.Series(pd.to_numeric(df["OBS_VALUE"], errors="coerce").values,
                  index=[to_period(t) for t in df["TIME_PERIOD"]]).dropna()
    return s.sort_index() if len(s) else None

In [ ]:
data = {}      # (block, variable, country) -> pandas Series
failures = []
inventory = ([("history", var, cc, key)
              for var, ccs in HISTORY_SERIES.items() for cc, key in ccs.items()]
             + [("mpd_forecast", item, cc, key)
                for item, ccs in MPD_SERIES.items() for cc, key in ccs.items()]
             + [("ameco", item, cc, key)
                for item, ccs in AME_SERIES.items() for cc, key in ccs.items()])

for i, (block, var, cc, key) in enumerate(inventory, 1):
    s = fetch_series(key)
    if s is None:
        failures.append(key)
    else:
        data[(block, var, cc)] = s
    if i % 50 == 0 or i == len(inventory):
        print(f"  {i}/{len(inventory)} fetched ({len(failures)} empty)")

print(f"\ndownloaded {len(data)} series; empty/missing: {len(failures)}")
if failures:
    print("missing:", failures)

  50/303 fetched (0 empty)
  100/303 fetched (0 empty)
  150/303 fetched (0 empty)


  100/303 fetched (0 empty)


  150/303 fetched (0 empty)


  200/303 fetched (0 empty)


  250/303 fetched (0 empty)


  300/303 fetched (0 empty)
  303/303 fetched (0 empty)

downloaded 303 series; empty/missing: 0


In [ ]:
example = data[("history", "unemployment", "ES")]
print("example — Spain, unemployment rate (monthly, % of labour force):")
print(example.tail(6).to_string())
print(f"\n{len(example)} observations, {example.index.min()} to {example.index.max()}")

---
## D — What did we get? Statistics

Per-series summaries below. Values are in each variable's native units — see
the variable dictionary in section A (`VARIABLE_GUIDE`).


In [ ]:
stats = pd.DataFrame([{
    "block": block, "variable": var, "country": cc,
    "freq": s.index.freqstr[0], "n_obs": len(s),
    "first": str(s.index.min()), "last": str(s.index.max()),
    "min": s.min(), "mean": s.mean(), "max": s.max(),
    "latest": s.iloc[-1],
} for (block, var, cc), s in data.items()])
print(f"{len(stats)} series | {stats.n_obs.sum():,} observations")
by_var = stats.groupby(["block", "variable"]).agg(
    countries=("country", "nunique"), obs=("n_obs", "sum"),
    earliest=("first", "min"), latest_end=("last", "max")).reset_index()
show(by_var, "Downloaded series by block and variable")

In [ ]:
coverage = (stats[stats.block == "history"]
            .assign(span=lambda d: d["first"].str[:4] + "-" + d["last"].str[:4]
                    + " (" + d.n_obs.astype(str) + ")")
            .pivot(index="country", columns="variable", values="span")
            .fillna("-"))
show(coverage, "History coverage: first-last (observations), native frequency")

In [ ]:
hist_stats = (stats[stats.block == "history"]
              [["variable", "country", "freq", "n_obs", "first", "last",
                "min", "mean", "max", "latest"]]
              .sort_values(["variable", "country"]).reset_index(drop=True))
(show(hist_stats, "Per-series statistics — history block")
 .format({"min": "{:,.2f}", "mean": "{:,.2f}", "max": "{:,.2f}",
          "latest": "{:,.2f}"})
 .background_gradient(subset=["n_obs"], cmap="Blues"))

### Official projections, current year onward

The tables below show the forward part of both forecast blocks: every year
from the **current year** to the projection horizon. MPD (December 2025
round) reaches **2028** for the euro-area countries; AMECO's EC forecasts
reach **2027** and cover the non-euro countries too (except Norway).


In [ ]:
mpd_titles = {"YER": "Real GDP growth, % (MPD)",
              "HIC": "HICP inflation, % (MPD)",
              "URX": "Unemployment rate, % (MPD)"}
for item, title in mpd_titles.items():
    block = {cc: s for (b, it, cc), s in data.items()
             if b == "mpd_forecast" and it == item}
    tbl = pd.DataFrame({cc: {p.year: v for p, v in s.items()
                             if p.year >= CURRENT_YEAR}
                        for cc, s in block.items()}).T.sort_index()
    tbl.columns.name, tbl.index.name = "year", "country"
    display(show(tbl.round(2), f"{title} — {MPD_ROUND} round, {CURRENT_YEAR} onward"))

In [ ]:
ame_titles = {"ZUTN": "Unemployment rate, % (AMECO/EC forecast)",
              "UDGGL": "Government gross debt, % of GDP (AMECO/EC forecast)",
              "UBLGE": "Government balance, % of GDP (AMECO/EC forecast)"}
for item, title in ame_titles.items():
    block = {cc: s for (b, it, cc), s in data.items()
             if b == "ameco" and it == item}
    tbl = pd.DataFrame({cc: {p.year: v for p, v in s.items()
                             if p.year >= CURRENT_YEAR}
                        for cc, s in block.items()}).T.sort_index()
    tbl.columns.name, tbl.index.name = "year", "country"
    display(show(tbl.round(2), f"{title} — {CURRENT_YEAR} onward"))

# AMECO real GDP is a level series -> show implied growth for readability
gdp = {cc: s for (b, it, cc), s in data.items() if b == "ameco" and it == "OVGD"}
growth = pd.DataFrame({cc: {p.year: v for p, v in
                            (s.pct_change() * 100).items()
                            if p.year >= CURRENT_YEAR}
                       for cc, s in gdp.items()}).T.sort_index()
growth.columns.name, growth.index.name = "year", "country"
show(growth.round(2),
     f"Real GDP growth implied by AMECO levels, % — {CURRENT_YEAR} onward")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
panels = [("unemployment", "Unemployment rate (%, monthly)", axes[0, 0]),
          ("hicp", "HICP inflation (% y/y, monthly)", axes[0, 1]),
          ("house_prices", "House prices (index, quarterly)", axes[1, 0]),
          ("sovereign_stress", "Sovereign stress indicator (0-1, monthly)", axes[1, 1])]
for var, title, ax in panels:
    for (block, v, cc), s in sorted(data.items()):
        if block == "history" and v == var:
            ax.plot(s.index.to_timestamp(), s.values, linewidth=0.9, label=cc)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=6, ncol=3, loc="upper left")
fig.suptitle("Selected history blocks — all countries", y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for (block, item, cc), s in sorted(data.items()):
    if block == "mpd_forecast" and item == "YER":
        tail = s[[p.year >= CURRENT_YEAR - 2 for p in s.index]]
        axes[0].plot([p.year for p in tail.index], tail.values, marker="o",
                     linewidth=1, label=cc)
    if block == "ameco" and item == "UDGGL":
        tail = s[[p.year >= CURRENT_YEAR - 6 for p in s.index]]
        axes[1].plot([p.year for p in tail.index], tail.values, marker="o",
                     linewidth=1, label=cc)
axes[0].set_title(f"MPD real GDP growth, % — {MPD_ROUND} round")
axes[1].set_title("Government debt, % of GDP (AMECO, last years = EC forecast)")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=6, ncol=3)
plt.tight_layout()
plt.show()

---
### Notes & attribution

- **Source:** European Central Bank — [ECB Data Portal](https://data.ecb.europa.eu),
  © ECB. Reuse is permitted subject to the
  [ECB's terms](https://www.ecb.europa.eu/services/disclaimer/html/index.en.html);
  please cite the ECB as source. `MNA`/`LFSI` originate from Eurostat.
- **Vintages:** history reflects the latest revisions at run time; the MPD
  projection round is pinned (reproducible), while AMECO forecasts are
  replaced silently at each EC publication — record your run date.
- Series keys were validated against the datasets' official series
  catalogues (section A links) at the time of writing; catalogues evolve, so
  re-run section B to re-validate.
